# 7 - Capa Serving (opcional)

Sumariza la data de BATCH (últimos 10 minutos, por `order_date`) junto con la data de SPEED (micro-batches procesados con trigger de 2 minutos).

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, expr, sum as _sum, count as _count

spark = (
    SparkSession.builder
    .appName("LambdaServing")
    .config("spark.jars", "/opt/spark/jars/mysql-connector-j-8.4.0.jar")
    .getOrCreate()
)

BATCH_PATH = "hdfs://namenode:8020/lambda/cleansed/batch/ventas"
SPEED_PATH = "hdfs://namenode:8020/lambda/speed"
SERVING_PATH = "hdfs://namenode:8020/lambda/serving"

## 1 - Leer capa BATCH (últimos 10 minutos)

In [10]:
ventas_batch = spark.read.parquet(BATCH_PATH)

ventas_batch_recientes = (
    ventas_batch
    .filter(expr("order_date >= current_timestamp() - INTERVAL 6 HOURS"))
    .select(
        "order_id",
        "order_customer_id",
        "order_status",
        "order_item_product_id",
        "order_item_quantity",
        "order_item_subtotal"
    )
    .withColumn("fuente", lit("BATCH"))
)

print("Registros BATCH (ultimos 10 min):", ventas_batch_recientes.count())
ventas_batch_recientes.show(5)

Registros BATCH (ultimos 10 min): 2617
+--------+-----------------+------------+---------------------+-------------------+-------------------+------+
|order_id|order_customer_id|order_status|order_item_product_id|order_item_quantity|order_item_subtotal|fuente|
+--------+-----------------+------------+---------------------+-------------------+-------------------+------+
|   69042|            10315|     PENDING|                  147|                  2|              43.98| BATCH|
|   69048|             4813|      CLOSED|                  391|                  2|             179.98| BATCH|
|   69048|             4813|      CLOSED|                  711|                  4|             199.96| BATCH|
|   69048|             4813|      CLOSED|                  150|                  1|             149.99| BATCH|
|   69352|              177|    COMPLETE|                  144|                  2|             499.98| BATCH|
+--------+-----------------+------------+---------------------+----------

## 2 - Leer capa SPEED

Nota: la tabla SPEED no persiste `order_date` (se descartó en el select del streaming), por lo que se lee el contenido acumulado escrito por los micro-batches de 2 minutos.

In [11]:
ventas_speed = (
    spark.read.parquet(SPEED_PATH)
    .select(
        "order_id",
        "order_customer_id",
        "order_status",
        "order_item_product_id",
        "order_item_quantity",
        "order_item_subtotal"
    )
    .withColumn("fuente", lit("SPEED"))
)

print("Registros SPEED:", ventas_speed.count())
ventas_speed.show(5)

Registros SPEED: 0
+--------+-----------------+------------+---------------------+-------------------+-------------------+------+
|order_id|order_customer_id|order_status|order_item_product_id|order_item_quantity|order_item_subtotal|fuente|
+--------+-----------------+------------+---------------------+-------------------+-------------------+------+
+--------+-----------------+------------+---------------------+-------------------+-------------------+------+



## 3 - Unificar BATCH + SPEED

In [12]:
ventas_unificadas = ventas_batch_recientes.unionByName(ventas_speed)

print("Total registros unificados:", ventas_unificadas.count())
ventas_unificadas.show(10)

Total registros unificados: 2617
+--------+-----------------+------------+---------------------+-------------------+-------------------+------+
|order_id|order_customer_id|order_status|order_item_product_id|order_item_quantity|order_item_subtotal|fuente|
+--------+-----------------+------------+---------------------+-------------------+-------------------+------+
|   69042|            10315|     PENDING|                  147|                  2|              43.98| BATCH|
|   69048|             4813|      CLOSED|                  391|                  2|             179.98| BATCH|
|   69048|             4813|      CLOSED|                  711|                  4|             199.96| BATCH|
|   69048|             4813|      CLOSED|                  150|                  1|             149.99| BATCH|
|   69352|              177|    COMPLETE|                  144|                  2|             499.98| BATCH|
|   69352|              177|    COMPLETE|                   51|                

## 4 - Sumarizacion

In [13]:
resumen_por_fuente = (
    ventas_unificadas
    .groupBy("fuente")
    .agg(
        _count("order_id").alias("total_items"),
        _sum("order_item_subtotal").alias("total_ventas")
    )
)
resumen_por_fuente.show()

resumen_por_status = (
    ventas_unificadas
    .groupBy("order_status")
    .agg(
        _count("order_id").alias("total_items"),
        _sum("order_item_subtotal").alias("total_ventas")
    )
    .orderBy("order_status")
)
resumen_por_status.show()

resumen_por_producto = (
    ventas_unificadas
    .groupBy("order_item_product_id")
    .agg(
        _count("order_id").alias("total_items_vendidos"),
        _sum("order_item_subtotal").alias("total_ventas")
    )
    .orderBy(_sum("order_item_subtotal").desc())
)
resumen_por_producto.show(10)

+------+-----------+-----------------+
|fuente|total_items|     total_ventas|
+------+-----------+-----------------+
| BATCH|       2617|785074.3999999773|
+------+-----------+-----------------+

+------------+-----------+------------------+
|order_status|total_items|      total_ventas|
+------------+-----------+------------------+
|      CLOSED|        696|209043.12999999986|
|    COMPLETE|        666|196327.78000000014|
|     PENDING|        607|176116.98000000016|
|  PROCESSING|        648| 203586.5099999999|
+------------+-----------+------------------+

+---------------------+--------------------+-----------------+
|order_item_product_id|total_items_vendidos|     total_ventas|
+---------------------+--------------------+-----------------+
|                  689|                   5|          8999.85|
|                 1020|                   5|          8249.85|
|                  208|                   1|          7999.96|
|                 1048|                   3|          769

## 5 - Guardar resumen en HDFS (SERVING)

In [14]:
resumen_por_fuente.write.mode("overwrite").parquet(f"{SERVING_PATH}/resumen_por_fuente")
resumen_por_status.write.mode("overwrite").parquet(f"{SERVING_PATH}/resumen_por_status")
resumen_por_producto.write.mode("overwrite").parquet(f"{SERVING_PATH}/resumen_por_producto")

print("Resumen guardado en", SERVING_PATH)

Resumen guardado en hdfs://namenode:8020/lambda/serving


In [15]:
print("BATCH raw count:", spark.read.parquet(BATCH_PATH).count())
spark.read.parquet(BATCH_PATH).select("order_date").orderBy(spark.read.parquet(BATCH_PATH)["order_date"].desc()).show(5, truncate=False)

print("SPEED raw count:", spark.read.parquet(SPEED_PATH).count())
spark.read.parquet(SPEED_PATH).show(5)

BATCH raw count: 174815


AnalysisException: [MISSING_ATTRIBUTES.RESOLVED_ATTRIBUTE_APPEAR_IN_OPERATION] Resolved attribute(s) "order_date" missing from "order_date" in operator !Sort [order_date#930 DESC NULLS LAST], true. Attribute(s) with the same name appear in the operation: "order_date".
Please check if the right attribute(s) are used.;
!Sort [order_date#930 DESC NULLS LAST], true
+- Project [order_date#908]
   +- Relation [order_id#907,order_date#908,order_customer_id#909,order_status#910,order_item_id#911,order_item_order_id#912,order_item_product_id#913,order_item_quantity#914,order_item_subtotal#915,order_item_product_price#916] parquet


In [ ]:
df_batch = spark.read.parquet(BATCH_PATH)
print("BATCH raw count:", df_batch.count())
df_batch.select("order_date").orderBy(df_batch["order_date"].desc()).show(5, truncate=False)

df_speed = spark.read.parquet(SPEED_PATH)
print("SPEED raw count:", df_speed.count())
df_speed.show(5)